In [ ]:
# @title 1. GPU and workspace check
from pathlib import Path
import os, platform, shutil, subprocess, sys

print("Python:", platform.python_version())
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun.") from e

ROOT = Path("/content/spinopelvic_fullres_fresh")
UPLOAD = ROOT / "00_uploaded_inputs"
UPLOAD.mkdir(parents=True, exist_ok=True)
print("Upload folder:", UPLOAD)
print("Free /content GiB:", round(shutil.disk_usage("/content").free / 2**30, 2))


In [ ]:
# @title 2. Upload fresh NRRD/NIfTI files
from google.colab import files
from pathlib import Path
import shutil

UPLOAD = Path("/content/spinopelvic_fullres_fresh/00_uploaded_inputs")
UPLOAD.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    src = Path("/content") / name
    dst = UPLOAD / name
    if src.exists() and src != dst:
        shutil.move(str(src), str(dst))
print("Files ready:")
for p in sorted(UPLOAD.glob("*")):
    print(p, p.stat().st_size)


In [ ]:
# @title 3. Save the full runner script
from pathlib import Path

RUNNER = "#!/usr/bin/env python3\n\"\"\"Full-resolution spinopelvic-seg Colab runner for uploaded NRRD/NIfTI CT files.\n\nResearch-only workflow. Outputs are segmentation candidates for human review, not\nclinical diagnoses.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport getpass\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport textwrap\nimport time\nimport zipfile\nfrom pathlib import Path\n\n\nRUN_ROOT = Path(\"/content/spinopelvic_fullres_fresh\")\nINPUT_ROOT = RUN_ROOT / \"00_uploaded_inputs\"\nNIFTI_ROOT = RUN_ROOT / \"01_converted_nifti\"\nNNINPUT_ROOT = RUN_ROOT / \"02_nninput\"\nMODEL_ROOT = RUN_ROOT / \"03_model\"\nNNUNET_RAW = RUN_ROOT / \"04_nnunet_raw\"\nNNUNET_PREPROCESSED = RUN_ROOT / \"05_nnunet_preprocessed\"\nNNUNET_RESULTS = RUN_ROOT / \"06_nnunet_results\"\nPRED_ROOT = RUN_ROOT / \"07_predictions\"\nREPORT_ROOT = RUN_ROOT / \"08_reports\"\nEXPORT_ROOT = RUN_ROOT / \"09_export\"\nLOG_ROOT = RUN_ROOT / \"logs\"\nSOURCE_ROOT = RUN_ROOT / \"source\" / \"spinopelvic-seg\"\n\nMODEL_REPO = \"anonymous-neurips-ED/spinopelvic-seg-checkpoints\"\nCODE_REPO = \"https://github.com/anonymous-mlhc/spinopelvic-seg.git\"\nDATASET_NAME = \"Dataset803_SpineSurgCTFullMerged\"\n\nLABELS = {\n    0: \"background\",\n    1: \"L1\",\n    2: \"L2\",\n    3: \"L3\",\n    4: \"L4\",\n    5: \"last_lumbar\",\n    6: \"sacrum\",\n    7: \"left_hip\",\n    8: \"right_hip\",\n    9: \"ignore_or_auxiliary\",\n}\n\nCOLORS = {\n    1: \"#e41a1c\",\n    2: \"#377eb8\",\n    3: \"#4daf4a\",\n    4: \"#984ea3\",\n    5: \"#ff7f00\",\n    6: \"#ffff33\",\n    7: \"#a65628\",\n    8: \"#f781bf\",\n    9: \"#999999\",\n}\n\nLIMITS = (\n    \"Research-only spinopelvic segmentation candidate. This does not diagnose \"\n    \"LSTV/Castellvi class or replace radiology review. Verify numbering and \"\n    \"left/right laterality anatomically, especially for cropped or nonstandard CT.\"\n)\n\n\ndef run(cmd: list[str], *, env: dict[str, str] | None = None, cwd: Path | None = None) -> None:\n    print(\"+\", \" \".join(cmd), flush=True)\n    subprocess.run(cmd, check=True, env=env, cwd=str(cwd) if cwd else None)\n\n\ndef capture(cmd: list[str], *, env: dict[str, str] | None = None, cwd: Path | None = None) -> str:\n    print(\"+\", \" \".join(cmd), flush=True)\n    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT, env=env, cwd=str(cwd) if cwd else None)\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\"rb\") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\"\"):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef case_id(index: int) -> str:\n    return f\"CASE_{index:03d}\"\n\n\ndef safe_name(path: Path) -> str:\n    clean = \"\".join(ch if ch.isalnum() or ch in \"._-\" else \"_\" for ch in path.stem)\n    return clean[:80] or \"ct\"\n\n\ndef prepare_dirs() -> None:\n    for path in (\n        RUN_ROOT,\n        INPUT_ROOT,\n        NIFTI_ROOT,\n        NNINPUT_ROOT,\n        MODEL_ROOT,\n        NNUNET_RAW,\n        NNUNET_PREPROCESSED,\n        NNUNET_RESULTS,\n        PRED_ROOT,\n        REPORT_ROOT,\n        EXPORT_ROOT,\n        LOG_ROOT,\n        SOURCE_ROOT.parent,\n    ):\n        path.mkdir(parents=True, exist_ok=True)\n\n\ndef install_environment(skip_install: bool) -> None:\n    if skip_install:\n        return\n    run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--upgrade\", \"pip\"])\n    run(\n        [\n            sys.executable,\n            \"-m\",\n            \"pip\",\n            \"install\",\n            \"-q\",\n            \"nnunetv2\",\n            \"huggingface_hub\",\n            \"SimpleITK\",\n            \"nibabel\",\n            \"pydicom\",\n            \"matplotlib\",\n            \"numpy\",\n            \"scipy\",\n            \"tqdm\",\n            \"gdown\",\n        ]\n    )\n\n\ndef clone_source() -> None:\n    if SOURCE_ROOT.exists() and any(SOURCE_ROOT.iterdir()):\n        print(f\"Using existing source checkout: {SOURCE_ROOT}\", flush=True)\n    else:\n        run([\"git\", \"clone\", CODE_REPO, str(SOURCE_ROOT)])\n    req = SOURCE_ROOT / \"requirements.txt\"\n    if req.exists():\n        run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"-r\", str(req)])\n\n\ndef login_hf(token: str | None) -> None:\n    if not token:\n        token = os.environ.get(\"HF_TOKEN\")\n    if token is None and sys.stdin.isatty():\n        token = getpass.getpass(\"Hugging Face token, or Enter if public access works: \").strip()\n    if token:\n        from huggingface_hub import login\n\n        login(token=token, add_to_git_credential=False)\n\n\ndef download_model() -> None:\n    from huggingface_hub import snapshot_download\n\n    print(f\"Downloading/checking model: {MODEL_REPO}\", flush=True)\n    snapshot_download(repo_id=MODEL_REPO, repo_type=\"model\", local_dir=str(MODEL_ROOT))\n    target = NNUNET_RESULTS / DATASET_NAME\n    source = MODEL_ROOT / DATASET_NAME\n    if target.exists():\n        shutil.rmtree(target)\n    if source.exists():\n        shutil.copytree(source, target)\n    else:\n        shutil.copytree(MODEL_ROOT, target)\n    (RUN_ROOT / \"model_manifest.json\").write_text(\n        json.dumps(\n            {\n                \"repo\": MODEL_REPO,\n                \"dataset_target\": str(target),\n                \"files\": {str(p.relative_to(MODEL_ROOT)): sha256(p) for p in MODEL_ROOT.rglob(\"*\") if p.is_file()},\n            },\n            indent=2,\n        )\n    )\n\n\ndef download_drive_folder(url: str | None, folder_id: str | None) -> None:\n    if not url and not folder_id:\n        return\n    import gdown\n\n    target = INPUT_ROOT / \"drive_folder\"\n    target.mkdir(parents=True, exist_ok=True)\n    if list(target.glob(\"*.nrrd\")) or list(target.glob(\"*.nii\")) or list(target.glob(\"*.nii.gz\")):\n        print(f\"Reusing existing CT files in {target}\", flush=True)\n        return\n    if url:\n        print(f\"Downloading CT folder from Google Drive into {target}\", flush=True)\n        gdown.download_folder(url=url, output=str(target), quiet=False, use_cookies=False)\n    else:\n        folder_url = f\"https://drive.google.com/drive/folders/{folder_id}\"\n        print(f\"Downloading CT folder from Google Drive into {target}\", flush=True)\n        gdown.download_folder(url=folder_url, output=str(target), quiet=False, use_cookies=False)\n\n\ndef find_inputs(explicit: list[str]) -> list[Path]:\n    paths = [Path(x) for x in explicit]\n    if not paths:\n        drive_root = INPUT_ROOT / \"drive_folder\"\n        roots = [drive_root] if drive_root.exists() and any(drive_root.rglob(\"*\")) else [INPUT_ROOT, Path(\"/content\"), Path(\"/content/drive/MyDrive\")]\n        for root in roots:\n            if root.exists():\n                for pattern in (\"*.nrrd\", \"*.nii\", \"*.nii.gz\"):\n                    paths.extend(root.rglob(pattern))\n    seen: set[Path] = set()\n    out: list[Path] = []\n    for path in paths:\n        path = path.expanduser()\n        if path.exists() and path.is_file() and path.resolve() not in seen:\n            seen.add(path.resolve())\n            out.append(path)\n    if not out:\n        raise FileNotFoundError(\"No NRRD/NIfTI files found. Upload files into /content/spinopelvic_fullres_fresh/00_uploaded_inputs.\")\n    return sorted(out, key=lambda p: p.name)\n\n\ndef convert_inputs(inputs: list[Path]) -> list[dict[str, object]]:\n    import numpy as np\n    import SimpleITK as sitk\n\n    shutil.rmtree(NNINPUT_ROOT, ignore_errors=True)\n    NNINPUT_ROOT.mkdir(parents=True, exist_ok=True)\n    rows: list[dict[str, object]] = []\n    for index, source in enumerate(inputs):\n        cid = case_id(index)\n        out = NIFTI_ROOT / f\"{cid}_{safe_name(source)}.nii.gz\"\n        nn_case = NNINPUT_ROOT / f\"{cid}_0000.nii.gz\"\n        img = sitk.ReadImage(str(source))\n        arr = sitk.GetArrayViewFromImage(img)\n        if img.GetDimension() != 3 or img.GetNumberOfComponentsPerPixel() != 1:\n            raise ValueError(f\"{source} is not a scalar 3D CT\")\n        if not np.isfinite(arr).all() or np.ptp(arr) == 0:\n            raise ValueError(f\"{source} is empty/constant/non-finite\")\n        for key in img.GetMetaDataKeys():\n            img.EraseMetaData(key)\n        sitk.WriteImage(img, str(out), True)\n        shutil.copy2(out, nn_case)\n        rows.append(\n            {\n                \"case_id\": cid,\n                \"source\": str(source),\n                \"converted\": str(out),\n                \"nnunet_input\": str(nn_case),\n                \"bytes\": source.stat().st_size,\n                \"sha256\": sha256(source),\n                \"size_xyz\": list(img.GetSize()),\n                \"spacing_xyz_mm\": list(img.GetSpacing()),\n                \"pixel_type\": img.GetPixelIDTypeAsString(),\n                \"min\": float(arr.min()),\n                \"max\": float(arr.max()),\n                \"nonzero_voxels\": int(np.count_nonzero(arr)),\n            }\n        )\n    (RUN_ROOT / \"input_manifest.json\").write_text(json.dumps(rows, indent=2))\n    with (RUN_ROOT / \"input_manifest.csv\").open(\"w\", newline=\"\") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    return rows\n\n\ndef nnunet_env() -> dict[str, str]:\n    env = os.environ.copy()\n    env.update(\n        {\n            \"nnUNet_raw\": str(NNUNET_RAW),\n            \"nnUNet_preprocessed\": str(NNUNET_PREPROCESSED),\n            \"nnUNet_results\": str(NNUNET_RESULTS),\n            \"MPLCONFIGDIR\": str(RUN_ROOT / \"matplotlib\"),\n            \"nnUNet_extTrainer\": str(SOURCE_ROOT),\n        }\n    )\n    Path(env[\"MPLCONFIGDIR\"]).mkdir(parents=True, exist_ok=True)\n    return env\n\n\ndef run_prediction(force: bool, device: str) -> None:\n    base_cmd = [\n        \"nnUNetv2_predict\", \"-d\", \"803\", \"-c\", \"3d_fullres\",\n        \"-p\", \"nnUNetResEncUNetPlans_100G\",\n        \"-tr\", \"nnUNetTrainerWandB_500ep_LSTVOversample\",\n        \"-f\", \"0\", \"-chk\", \"checkpoint_best.pth\",\n        \"-npp\", \"1\", \"-nps\", \"1\", \"-device\", device,\n        \"--disable_tta\", \"--continue_prediction\",\n    ]\n    log_path = LOG_ROOT / \"nnunet_predict.log\"\n    env = nnunet_env()\n    failures = []\n    with log_path.open(\"a\") as log:\n        for case_file in sorted(NNINPUT_ROOT.glob(\"CASE_*_0000.nii.gz\")):\n            cid = case_file.name.split(\"_0000\")[0]\n            out_file = PRED_ROOT / f\"{cid}.nii.gz\"\n            if out_file.exists() and not force:\n                print(f\"Reusing prediction: {out_file}\", flush=True)\n                continue\n            temp_in = RUN_ROOT / \"single_case_in\" / cid\n            shutil.rmtree(temp_in, ignore_errors=True)\n            temp_in.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(case_file, temp_in / case_file.name)\n            cmd = base_cmd + [\"-i\", str(temp_in), \"-o\", str(PRED_ROOT)]\n            log.write(\"\\n\\n### \" + cid + \"\\n\" + \" \".join(cmd) + \"\\n\")\n            log.flush()\n            print(f\"Predicting {cid}\", flush=True)\n            proc = subprocess.run(cmd, env=env, text=True, stdout=log, stderr=subprocess.STDOUT)\n            if proc.returncode != 0:\n                failures.append({\"case_id\": cid, \"returncode\": proc.returncode})\n                print(f\"FAILED {cid}; continuing\", flush=True)\n            else:\n                print(f\"Done {cid}\", flush=True)\n    if failures:\n        (LOG_ROOT / \"prediction_failures.json\").write_text(json.dumps(failures, indent=2))\n        print(f\"{len(failures)} nnUNet case(s) failed; rendering successful cases anyway\", flush=True)\n\n\ndef render_reports(manifest: list[dict[str, object]]) -> None:\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    import nibabel as nib\n    import numpy as np\n    from matplotlib.colors import BoundaryNorm, ListedColormap\n    from scipy import ndimage\n\n    colors = [\"#000000\"] + [COLORS.get(i, \"#ffffff\") for i in range(1, max(LABELS) + 1)]\n    cmap = ListedColormap(colors)\n    norm = BoundaryNorm(np.arange(-0.5, len(colors) + 0.5), len(colors))\n    all_rows: list[dict[str, object]] = []\n\n    for row in manifest:\n        cid = str(row[\"case_id\"])\n        ct_path = Path(str(row[\"converted\"]))\n        pred_path = PRED_ROOT / f\"{cid}.nii.gz\"\n        if not pred_path.exists():\n            pred_path = PRED_ROOT / f\"{cid}_0000.nii.gz\"\n        if not pred_path.exists():\n            print(f\"Skipping {cid}: missing prediction\", flush=True)\n            continue\n        case_dir = REPORT_ROOT / cid\n        img_dir = case_dir / \"overlays\"\n        mask_dir = case_dir / \"masks\"\n        img_dir.mkdir(parents=True, exist_ok=True)\n        mask_dir.mkdir(parents=True, exist_ok=True)\n\n        ct_img = nib.load(str(ct_path))\n        pred_img = nib.load(str(pred_path))\n        if ct_img.shape != pred_img.shape:\n            raise ValueError(f\"{cid} shape mismatch: CT {ct_img.shape} vs prediction {pred_img.shape}\")\n        ct = np.asanyarray(nib.as_closest_canonical(ct_img).dataobj)\n        pred_c = nib.as_closest_canonical(pred_img)\n        pred = np.asanyarray(pred_c.dataobj).astype(np.int16)\n        present = [int(x) for x in np.unique(pred) if int(x) > 0]\n        voxel_ml = abs(np.linalg.det(pred_img.affine[:3, :3])) / 1000\n        stats_rows = []\n        for lab in present:\n            mask = pred == lab\n            count = int(mask.sum())\n            if not count:\n                continue\n            nib.save(\n                nib.Nifti1Image(mask.astype(\"uint8\"), pred_c.affine, pred_c.header),\n                mask_dir / f\"{lab:02d}_{LABELS.get(lab, 'label')}.nii.gz\",\n            )\n            center = ndimage.center_of_mass(mask)\n            stats = {\n                \"case_id\": cid,\n                \"label_id\": lab,\n                \"label\": LABELS.get(lab, \"unknown\"),\n                \"voxels\": count,\n                \"volume_ml\": round(count * voxel_ml, 3),\n                \"tags\": \"prediction; requires_human_review\",\n            }\n            stats_rows.append(stats)\n            all_rows.append(stats)\n\n        planes = [(\"sagittal\", 0), (\"coronal\", 1), (\"axial\", 2)]\n        for plane, axis in planes:\n            occupancy = np.count_nonzero(pred, axis=tuple(a for a in range(3) if a != axis))\n            index = int(occupancy.argmax()) if occupancy.max() else pred.shape[axis] // 2\n            image = np.take(ct, index, axis=axis).T\n            mask = np.take(pred, index, axis=axis).T\n            fig, ax = plt.subplots(figsize=(9, 9), layout=\"constrained\")\n            ax.imshow(image, cmap=\"gray\", vmin=-500, vmax=1300, origin=\"lower\")\n            ax.imshow(np.ma.masked_equal(mask, 0), cmap=cmap, norm=norm, alpha=0.48, origin=\"lower\", interpolation=\"nearest\")\n            for lab in sorted(np.unique(mask)):\n                if lab <= 0:\n                    continue\n                yy, xx = ndimage.center_of_mass(mask == lab)\n                ax.text(\n                    xx,\n                    yy,\n                    LABELS.get(int(lab), str(int(lab))),\n                    color=\"white\",\n                    fontsize=9,\n                    ha=\"center\",\n                    bbox={\"facecolor\": COLORS.get(int(lab), \"#333333\"), \"alpha\": 0.85, \"edgecolor\": \"white\", \"pad\": 2},\n                )\n            ax.set_title(f\"{cid} {plane} overlay slice {index}\")\n            ax.axis(\"off\")\n            fig.savefig(img_dir / f\"{cid}_{plane}_ct_overlay.png\", dpi=180)\n            plt.close(fig)\n\n        (case_dir / \"case_report.json\").write_text(\n            json.dumps(\n                {\n                    \"case_id\": cid,\n                    \"source\": row[\"source\"],\n                    \"ct\": str(ct_path),\n                    \"prediction\": str(pred_path),\n                    \"labels\": stats_rows,\n                    \"limitations\": LIMITS,\n                },\n                indent=2,\n            )\n        )\n        with (case_dir / \"label_stats.csv\").open(\"w\", newline=\"\") as f:\n            writer = csv.DictWriter(f, fieldnames=[\"case_id\", \"label_id\", \"label\", \"voxels\", \"volume_ml\", \"tags\"])\n            writer.writeheader()\n            writer.writerows(stats_rows)\n\n    if all_rows:\n        with (REPORT_ROOT / \"all_case_label_stats.csv\").open(\"w\", newline=\"\") as f:\n            writer = csv.DictWriter(f, fieldnames=[\"case_id\", \"label_id\", \"label\", \"voxels\", \"volume_ml\", \"tags\"])\n            writer.writeheader()\n            writer.writerows(all_rows)\n    (REPORT_ROOT / \"label_schema.json\").write_text(json.dumps({str(k): v for k, v in LABELS.items()}, indent=2))\n    (REPORT_ROOT / \"LIMITATIONS.txt\").write_text(textwrap.fill(LIMITS, 88) + \"\\n\")\n\n\ndef zip_results() -> Path:\n    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)\n    archive = EXPORT_ROOT / \"Spinopelvic_Fullres_Fresh_RESULTS.zip\"\n    if archive.exists():\n        archive.unlink()\n    with zipfile.ZipFile(archive, \"w\", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:\n        for root in (PRED_ROOT, REPORT_ROOT, LOG_ROOT):\n            for path in sorted(root.rglob(\"*\")):\n                if path.is_file():\n                    z.write(path, path.relative_to(RUN_ROOT))\n        for path in (RUN_ROOT / \"input_manifest.json\", RUN_ROOT / \"input_manifest.csv\", RUN_ROOT / \"model_manifest.json\"):\n            if path.exists():\n                z.write(path, path.relative_to(RUN_ROOT))\n    with zipfile.ZipFile(archive) as z:\n        bad = z.testzip()\n        if bad:\n            raise RuntimeError(f\"ZIP integrity failed at {bad}\")\n    (EXPORT_ROOT / \"ZIP_SHA256.txt\").write_text(f\"{sha256(archive)}  {archive.name}\\n\")\n    return archive\n\n\ndef zip_single_case(case_id: str) -> Path:\n    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)\n    archive = EXPORT_ROOT / f\"{case_id}_Spinopelvic_RESULT.zip\"\n    if archive.exists():\n        archive.unlink()\n    with zipfile.ZipFile(archive, \"w\", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:\n        for root in (PRED_ROOT, REPORT_ROOT / case_id, LOG_ROOT):\n            if root.exists():\n                for path in sorted(root.rglob(\"*\")):\n                    if path.is_file():\n                        z.write(path, path.relative_to(RUN_ROOT))\n        for path in (RUN_ROOT / \"input_manifest.json\", RUN_ROOT / \"input_manifest.csv\", RUN_ROOT / \"model_manifest.json\"):\n            if path.exists():\n                z.write(path, path.relative_to(RUN_ROOT))\n    with zipfile.ZipFile(archive) as z:\n        bad = z.testzip()\n        if bad:\n            raise RuntimeError(f\"ZIP integrity failed at {bad}\")\n    return archive\n\n\ndef main() -> None:\n    # Colab's launcher can leak its own flags into sys.argv when this script is\n    # executed through `colab exec -f`; keep only arguments meant for this file.\n    if Path(sys.argv[0]).name.startswith(\"colab_kernel_launcher\"):\n        sys.argv = [__file__]\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"inputs\", nargs=\"*\", help=\"NRRD/NIfTI CT paths. If omitted, auto-searches upload folders.\")\n    parser.add_argument(\"--hf-token\", default=None, help=\"Hugging Face token. Prefer HF_TOKEN env var or getpass in notebook.\")\n    parser.add_argument(\"--drive-folder-url\", default=None, help=\"Public/shared Google Drive folder URL containing CT NRRD/NIfTI files.\")\n    parser.add_argument(\"--drive-folder-id\", default=None, help=\"Google Drive folder ID containing CT NRRD/NIfTI files.\")\n    parser.add_argument(\"--only-case\", default=None, help=\"Run/report one converted case id such as CASE_006.\")\n    parser.add_argument(\"--device\", default=\"cuda\", choices=[\"cuda\", \"cpu\"], help=\"nnUNet inference device.\")\n    parser.add_argument(\"--skip-install\", action=\"store_true\")\n    parser.add_argument(\"--force\", action=\"store_true\", help=\"Rerun prediction even if predictions exist.\")\n    args = parser.parse_args()\n\n    started = time.time()\n    prepare_dirs()\n    install_environment(args.skip_install)\n    clone_source()\n    login_hf(args.hf_token)\n    download_model()\n    download_drive_folder(args.drive_folder_url, args.drive_folder_id)\n    inputs = find_inputs(args.inputs)\n    manifest = convert_inputs(inputs)\n    if args.only_case:\n        keep = [r for r in manifest if r[\"case_id\"] == args.only_case]\n        if not keep:\n            raise ValueError(f\"Requested {args.only_case}, but manifest has {[r['case_id'] for r in manifest]}\")\n        for p in NNINPUT_ROOT.glob(\"CASE_*_0000.nii.gz\"):\n            if not p.name.startswith(args.only_case + \"_\"):\n                p.unlink()\n        manifest = keep\n    run_prediction(force=args.force, device=args.device)\n    render_reports(manifest)\n    if args.only_case:\n        archive = zip_single_case(args.only_case)\n        status = {\n            \"status\": \"single_case_complete_or_partial\",\n            \"archive\": str(archive),\n            \"archive_sha256\": sha256(archive),\n            \"case_count\": 1,\n            \"limitations\": LIMITS,\n        }\n        (RUN_ROOT / f\"{args.only_case}_run_status.json\").write_text(json.dumps(status, indent=2))\n        print(json.dumps(status, indent=2), flush=True)\n        return\n    archive = zip_results()\n    status = {\n        \"status\": \"complete_with_possible_case_failures\" if (LOG_ROOT / \"prediction_failures.json\").exists() else \"complete\",\n        \"archive\": str(archive),\n        \"archive_sha256\": sha256(archive),\n        \"case_count\": len(manifest),\n        \"elapsed_seconds\": round(time.time() - started, 2),\n        \"limitations\": LIMITS,\n    }\n    (RUN_ROOT / \"run_status.json\").write_text(json.dumps(status, indent=2))\n    print(json.dumps(status, indent=2), flush=True)\n    try:\n        from google.colab import files\n\n        files.download(str(archive))\n    except Exception:\n        print(f\"Download ZIP manually from: {archive}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"
path = Path("/content/spinopelvic_fullres_fresh/spinopelvic_fullres_nrrd_colab.py")
path.write_text(RUNNER)
print("Wrote:", path)


In [ ]:
# @title 4. Run full-resolution spinopelvic segmentation
import getpass, os, runpy, sys

token = getpass.getpass("Hugging Face token, or Enter if public access works: ").strip()
if token:
    os.environ["HF_TOKEN"] = token

sys.argv = ["/content/spinopelvic_fullres_fresh/spinopelvic_fullres_nrrd_colab.py"]
runpy.run_path(sys.argv[0], run_name="__main__")


In [ ]:
# @title 5. Download final ZIP again if needed
from google.colab import files
from pathlib import Path

zip_path = Path("/content/spinopelvic_fullres_fresh/09_export/Spinopelvic_Fullres_Fresh_RESULTS.zip")
print(zip_path, zip_path.exists(), zip_path.stat().st_size if zip_path.exists() else None)
files.download(str(zip_path))
